In [0]:
%run "../00_config"

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

##Load cities from silver orders and weather

In [0]:
cities_orders = spark.table(SILVER_ORDERS).select("city").distinct()
cities_weather = spark.table(SILVER_WEATHER).select("city", "latitude", "longitude").distinct()
display(cities_weather)

##Build dim_city

In [0]:
# Union cities from both sources then join coordinates from weather
all_cities = cities_orders.union(
    cities_weather.select("city")
).distinct()

# Join coordinates from silver weather
df_dim_city = (all_cities
    .join(
        cities_weather.groupBy("city").agg(
            F.avg("latitude").alias("latitude"),
            F.avg("longitude").alias("longitude")
        ),
        on="city",
        how="left"
    )
    .filter(F.col("city").isNotNull())
    .filter(F.col("city") != "")

    # Surrogate key
    .withColumn("city_id",
        F.concat(F.lit("CITY_"), F.upper(F.col("city")))
    )

    .withColumn("country",      F.lit("Saudi Arabia"))
    .withColumn("country_code", F.lit("SA"))

    # Ingestion metadata
    .withColumn("_gold_ingested_at",
        F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat()))
    )

    .select(
        "city_id",
        "city",
        "country",
        "latitude",
        "longitude",
        "_gold_ingested_at"
    )
)

print(f"dim_city rows: {df_dim_city.count()}")
display(df_dim_city)

##Write to gold

In [0]:
(df_dim_city
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_CITY)
)

print(f"✅ {df_dim_city.count()} rows written to {GOLD_CITY}")

##Validate

In [0]:
df_check = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_city")
print(f"Total rows: {df_check.count()}")
df_check.show(truncate=False)

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.gold.dim_city;